## Day 2 Loader

In [3]:
import pandas as pd
import logging
import yaml
from pathlib import Path
from typing import Optional

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class HRDataLoader:
    """
    Production-grade data loader for HR Talent Analytics.
    Handles ingestion of all 5 raw CSV files with error handling.
    """

    def __init__(self, config_path: str = 'config.yaml'):
        try:
            with open(config_path, 'r') as file:
                self.config = yaml.safe_load(file)
            logger.info("Config file loaded successfully")
        except Exception as e:
            logger.error(f"Failed to load config file: {e}")
            self.config = {}

        try:
            raw_path_str = self.config.get('data', {}).get('raw_path', 'data/raw')
            self.raw_path = Path(raw_path_str)
            logger.info(f"Raw data path set to: {self.raw_path}")
        except Exception as e:
            logger.error(f"Error setting raw data path: {e}")
            self.raw_path = Path('data/raw')

        self.employees = None
        self.projects = None
        self.performance = None
        self.training = None
        self.salary = None

    def load_all(self) -> dict:
        """Load all datasets and return as dictionary."""
        logger.info("Starting to load all datasets...")
        self.employees = self.load_employees()
        self.projects = self.load_projects()
        self.salary = self.load_salary()
        self.performance = self.load_performance()
        self.training = self.load_training()

        data = {
            "employees": self.employees,
            "projects": self.projects,
            "salary": self.salary,
            "performance": self.performance,
            "training": self.training
        }
        logger.info("All datasets loaded successfully")
        return data

    def load_employees(self) -> pd.DataFrame:
        """Load and return employees dataset."""
        return self._load_csv("employees.csv")

    def load_projects(self) -> pd.DataFrame:
        """Load and return project assignments dataset."""
        return self._load_csv("project_assignments.csv")

    def load_performance(self) -> pd.DataFrame:
        """Load and return performance reviews dataset."""
        return self._load_csv("performance_reviews.csv")

    def load_training(self) -> pd.DataFrame:
        """Load and return training history dataset."""
        return self._load_csv("training_history.csv")

    def load_salary(self) -> pd.DataFrame:
        """Load and return salary history dataset."""
        return self._load_csv("salary_history.csv")

    def _load_csv(self, filename: str) -> Optional[pd.DataFrame]:
        """
        Private method: load a single CSV with error handling.
        Log success or failure. Never crash silently.
        """
        try:
            filepath = self.raw_path / filename
            df = pd.read_csv(filepath)
            logger.info(f"Loaded {filename}: {df.shape[0]} rows, {df.shape[1]} columns")
            return df
        except FileNotFoundError:
            logger.error(f"File not found: {filename}")
            return None
        except Exception as e:
            logger.error(f"Failed to load {filename}: {e}")
            return None

In [8]:
import pandas as pd
import logging
import yaml
from pathlib import Path
from typing import Optional
from datetime import date
# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__) 

In [2]:
import pandas as pd
import logging
import yaml
from pathlib import Path
from typing import Optional

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__) 

class HRDataLoader:
    """
    Production-grade data loader for HR Talent Analytics.
    Handles ingestion of all 5 raw CSV files with error handling.
    """

    def __init__(self, config_path: str = 'config.yaml'):
        # Load config
        try: 
            with open(config_path, 'r') as file:
                self.config = yaml.safe_load(file) # attribute 
            logger.info("Config file loaded successfully")
        except Exception as e:
            logger.error(f"Failed to load config file: {e}")
            self.config ={}
       
        # Set raw data path
        try: 
            raw_path_str = self.config.get('data', {}).get('raw_path', 'data/raw')
            self.raw_path = Path(raw_path_str)
            logger.info(f"Raw data path set to: {self.raw_path}")
        except Exception as e:
            logger.error(f"Error setting raw data path: {e}")
            self.raw_path = Path('data/raw')
        
        # 3) Initialize empty datasets
        self.employees = None
        self.projects = None
        self.performance = None
        self.training = None
        self.salary = None
        

    def load_all(self) -> dict:
        """Load all datasets and return as dictionary."""

        logger.info("Starting to load all datasets...")
        self.employees = self.load_employees()
        self.projects = self.load_projects()
        self.salary = self.load_salary()
        self.performance = self.load_performance()
        self.training = self.load_training()

        # dict

        data = {
            "employees": self.employees, 
            "projects" : self.projects,
            "salary" : self.salary, 
            "performance": self.performance,
            "training": self.training
        }
        logger.info("All datasets loaded successfully")

        return data

    def load_employees(self) -> pd.DataFrame:
        """Load and return employees dataset."""
        return self._load_csv("employees.csv")
        
    def load_projects(self) -> pd.DataFrame:
        """Load and return project assignments dataset."""
        
        return self.load_csv("project_assignments.csv")

    def load_performance(self) -> pd.DataFrame:
        """Load and return performance reviews dataset."""
        return self.load_csv("performance_reviews.csv")
       

    def load_training(self) -> pd.DataFrame:
        """Load and return training history dataset."""
        return self.load_csv("training_history.csv")
        

    def load_salary(self) -> pd.DataFrame:
        """Load and return salary history dataset."""
        return self.load_csv("salary_history.csv")
       

    def _load_csv(self, filename: str) -> Optional[pd.DataFrame]:
        """
        Private method: load a single CSV with error handling.
        Log success or failure. Never crash silently.
        """
        try: 
            filepath = self.raw_path / filename
            df= pd.read_csv(filepath)
            logger.info(f"{filename} loaded successfully")
            return df
        except Exception as e:
            logger.error(f"Failed to load {filename}: {e}")
            None


In [ ]:
import pandas as pd
import logging
from typing import Tuple

logger = logging.getLogger(__name__)

class HRDataValidator:
    """
    Validates loaded HR datasets for quality and completeness.
    Generates a validation report for each dataset.
    """

    def validate_employees(self, df: pd.DataFrame) -> Tuple[bool, list]:
        """
        Validate employees dataset.
        Returns: (is_valid: bool, issues: list of strings)
        """
        issues = []
        if df['employee_id'].duplicated().sum() > 0: 
            issues.append('Duplicated founded')
        if df['employee_id'].isnull().sum() > 0:
            issues.append('Null values in Employee id')
        invalid_loc = ~df['Office_location'].isin(['Dhahran', 'Jeddah', 'Riyadh'])
        if  invalid_loc.any():
            issues.append("Invalid Office_location values found")

        is_valid = len(issues) == 0
        return is_valid, issues

            

    def validate_performance(self, df: pd.DataFrame) -> Tuple[bool, list]:
        """
        Validate performance reviews dataset.
        Returns: (is_valid: bool, issues: list of strings)
        """
        issues = []
        # check nulls
        if df['employee_id'].isnull().sum() > 0:
            issues.append('Null values in Employee id')
        # check intervels
        cols = ['technical_score', 'leadership_score', 'manager_rating']
        mask = ((df[cols] < 1) | (df[cols] > 5))
        if mask.any(): 
            issues.append('out of range score')
            
        future_rev = df['review_quarter'] > date.today()
        if future_rev.any():
            issues.append('future date')       
        
        is_valid = len(issues) == 0
        return is_valid, issues

    def generate_report(self, datasets: dict) -> None:
        """
        Run all validations and print a clean summary report.
        """
        # loop
        for k, v in datasets.items():
            print(f"Validating {k}...")
            issues = []
            if k == 'performance':
                issues.appened(self.validate_performance(v))
            elif k == 'employees':
                issues.appened(self.validate_employees(v))
        print(f"  Status: {'✅ Valid' if is_valid else '❌ Issues found'}")
        for issue in issues:
            print(f"  - {issue}")
        
        
        